In [ ]:
import pandas as pd
import numpy as np

import re
import html
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.listdir("/content/drive/MyDrive/Colab Notebooks/DuplicateDetection")

['MozillaCore', 'Firefox']

In [ ]:
firefox = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/DuplicateDetection/Firefox/mozilla_firefox.csv",
    low_memory=False
)

core = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/DuplicateDetection/MozillaCore/mozilla_core.csv",
    low_memory=False
)

In [ ]:
print("Firefox:", firefox.shape)
print("Core:", core.shape)

Firefox: (115814, 11)
Core: (205069, 11)


In [ ]:
firefox.head()
core.head()

,Issue_id,Priority,Component,Duplicated_issue,Title,Description,Status,Resolution,Version,Created_time,Resolved_time
0,91,P3,DOM: Core & HTML,NaN,document properties cannot be listed,Created by Till Krech (till@berlin.snafu.de) on Tuesday; April 7; 1998 9:05:23 AM PDT; Additional Details :; the JavaScript for in statement does not work on the; document object. At least not in ...,VERIFIED,FIXED,Trunk,1998-04-07 16:05:23 -0700,2009-02-16 08:04:06 -0800
1,103,P1,Layout: Tables,NaN,layout bug: table cell overflows containing cell,Created by Darrell Kindred (dkindred=mozilla@cs.cmu.edu) on Tuesday; April 7; 1998 11:43:26 AM PDT; Additional Details :; The nested table here is laid out wrong:; ; http://www.cs.cmu.edu/People/d...,VERIFIED,FIXED,Trunk,1998-04-07 18:43:26 -0700,2009-02-16 08:05:07 -0800
2,104,P3,Serializers,NaN,table saved as text: missing inter-column space,Created by Darrell Kindred (dkindred=mozilla@cs.cmu.edu) on Tuesday; April 7; 1998 12:16:02 PM PDT; Additional Details :; Visit this document (which contains a table);; and save as text:; ; http:/...,VERIFIED,WONTFIX,Trunk,1998-04-07 19:16:02 -0700,2007-04-06 22:26:22 -0700
3,105,P2,Layout: Tables,NaN,nested <TABLE>s: bgcolor of inner table not recognized,Created by (msuencks@marcant.de) on Tuesday; April 7; 1998 12:54:41 PM PDT; Additional Details :; The following HTML Code is not properley drawn:; ; <body bgcolor=white>; <table bgcolor=blue>; <...,VERIFIED,FIXED,Trunk,1998-04-07 19:54:41 -0700,2009-02-16 08:18:13 -0800
4,133,P3,Internationalization,NaN,Navigator draws entities like &lt;&amp;&gt; as ?s when KOI-8 encoding is set.,Created by Boris Tobotras (boris@xtalk.msk.su) on Thursday; April 9; 1998 9:41:16 PM PDT; Additional Details :; To reproduce is easy: point navigator to the HTML document:; <HTML><HEAD></HEAD><BOD...,VERIFIED,FIXED,Trunk,1998-04-10 04:41:16 -0700,2009-02-16 08:19:17 -0800


In [ ]:
print("=== FIREFOX ===")
for col in firefox.columns:
    print(col)

print("\n=== CORE ===")
for col in core.columns:
    print(col)

=== FIREFOX ===
Issue_id
Priority
Component
Duplicated_issue
Title
Description
Status
Resolution
Version
Created_time
Resolved_time

=== CORE ===
Issue_id
Priority
Component
Duplicated_issue
Title
Description
Status
Resolution
Version
Created_time
Resolved_time


In [ ]:
firefox.isna().sum().sort_values(ascending=False).head(20)

,0
Duplicated_issue,80000
Description,785
Title,4
Component,0
Priority,0
Issue_id,0
Status,0
Resolution,0
Version,0
Created_time,0


In [ ]:
firefox.sample(5)

,Issue_id,Priority,Component,Duplicated_issue,Title,Description,Status,Resolution,Version,Created_time,Resolved_time
80895,563351,--,General,212163.0,Popups triggered by clicking on transparent elements,User-Agent: Mozilla/5.0 (Windows; U; Windows NT 5.2; en-US; rv:1.9.2.3) Gecko/20100401 Firefox/3.6.3 (.NET CLR 3.5.30729); Build Identifier: Mozilla/5.0 (Windows; U; Windows NT 5.2; en-US; r...,RESOLVED,DUPLICATE,unspecified,2010-05-03 08:53:50 -0700,2010-05-03 09:01:27 -0700
30499,323446,--,General,NaN,Cyrilic fonts do not appear in textboxes; toolbars and and bookmarks in Mac OSX 10.4.4,User-Agent: Mozilla/5.0 (Macintosh; U; PPC Mac OS X Mach-O; en-US; rv:1.8) Gecko/20051111 Firefox/1.5; Build Identifier: Mozilla/5.0 (Macintosh; U; PPC Mac OS X Mach-O; en-US; rv:1.8) Gecko/...,RESOLVED,WORKSFORME,1.5.0.x Branch,2006-01-14 12:29:27 -0800,2007-06-28 00:51:27 -0700
25762,308103,--,Tabbed Browser,300198.0,Open in Tabs Behavior Change,User-Agent: Mozilla/5.0 (Windows; U; Windows NT 5.0; en-US; rv:1.8b4) Gecko/20050908 Firefox/1.4; Build Identifier: Mozilla/5.0 (Windows; U; Windows NT 5.0; en-US; rv:1.8b4) Gecko/20050908 F...,RESOLVED,DUPLICATE,unspecified,2005-09-11 22:11:24 -0700,2005-09-11 22:22:26 -0700
46948,393879,--,Bookmarks & History,NaN,view | show columns | name is not disabled,view | show columns | name is not disabled; ; the problem is:; ; // Cannot uncheck the primary column.; if (column.primary); menuitem.setAttribute(disabled; true);; ; we ...,VERIFIED,FIXED,Trunk,2007-08-27 11:45:24 -0700,2009-11-26 05:51:46 -0800
47471,396509,P1,General,NaN,Standard Vista account with Parental Controls turned on cannot launch Firefox,If a Standard account on Windows Vista has Parental Controls turned on for it at all; when that account tries to launch Firefox; the busy mouse pointer will appear for a second and then disappear....,VERIFIED,FIXED,Trunk,2007-09-17 18:10:19 -0700,2008-01-14 14:00:31 -0800


In [ ]:
firefox["project"] = "firefox"
core["project"] = "mozillacore"

In [ ]:
df = pd.concat(
    [firefox, core],
    ignore_index=True
)

print(df.shape)

(320883, 12)


In [ ]:
df = df.rename(columns={
    "Issue_id": "issue_id",
    "Priority": "priority",
    "Component": "component",
    "Duplicated_issue": "duplicate_of",
    "Title": "title",
    "Description": "description",
    "Status": "status",
    "Resolution": "resolution",
    "Version": "Version",
    "Created_time": "created_at",
    "Resolved_time": "resolved_at",

})

In [ ]:
columns_needed = [
    "issue_id",
    "title",
    "description",
    "created_at",
    "duplicate_of",
    "component",
    "resolution",
    "project"
]

df = df[columns_needed].copy()

In [ ]:
df.head()

,issue_id,title,description,created_at,duplicate_of,component,resolution,project
0,10954,Dialup properties needs to be exposed in prefs,The dialup properties of the profile should be exposed in the prefs panels so; the user has an easy way to modify them. The only other alternative would be to; make people go to the profile manag...,1999-07-30 15:55:51 -0700,NaN,Preferences,WONTFIX,firefox
1,14871,[Find] Find whole word only,Please add Match Whole Word Only option to browsers Find on this page; dialog.,1999-09-24 14:49:34 -0700,269442.0,General,DUPLICATE,firefox
2,19118,Plug-In Manager (ui for choosing mimetype-plugin associations),I would really like a plug-in manager for my browser that allows me to choose; which mime types are controlled by which plug-ins.; In the browser preferences window there should be a plug-in manag...,1999-11-17 14:58:26 -0800,NaN,Preferences,WONTFIX,firefox
3,54746,Language encodings in font prefs dialog not sorted,Language encodings are listed in a seemingly random order.; The order be alphabetical (and therefore change with localization).; As a special case; User-Defined should be last.,2000-09-29 14:12:11 -0700,NaN,Preferences,WORKSFORME,firefox
4,56892,Synaptics touchpad scrolling not working,From Bugzilla Helper:; User-Agent: Mozilla/5.0 (Windows; U; Win95; en-US; m18) Gecko/20001010; BuildID: 2000101014; ; I am using a synaptics touch pad with the latest Win 98 driver V 5.0.86 AUg...,2000-10-16 14:48:15 -0700,NaN,General,WORKSFORME,firefox


In [ ]:
df["created_at"] = pd.to_datetime(
    df["created_at"],
    errors="coerce",
    utc=True
)
if "resolved_at" in df.columns:
    df["resolved_at"] = pd.to_datetime(
        df["resolved_at"],
        errors="coerce",
        utc=True
    )

In [ ]:
before = len(df)

df = df.dropna(subset=["created_at"])

print("Removed:", before - len(df))
print("Remaining:", len(df))

Removed: 0
Remaining: 320883


In [ ]:
df["issue_id"].duplicated().sum()

np.int64(0)

In [ ]:
df["global_id"] = (
    df["project"].astype(str)
    + "::"
    + df["issue_id"].astype(str)
)

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x)

    # Decode HTML entities
    x = html.unescape(x)

    # Remove HTML tags
    x = re.sub(r"<[^>]+>", " ", x)

    # Replace URL
    x = re.sub(
        r"https?://\S+|www\.\S+",
        " <URL> ",
        x
    )

    # Normalize whitespace
    x = re.sub(r"\s+", " ", x)

    return x.strip()

In [ ]:
df["title_clean"] = df["title"].apply(normalize_text)
df["description_clean"] = df["description"].apply(normalize_text)

In [ ]:
df["text"] = (
    df["title_clean"]
    + " "
    + df["description_clean"]
).str.strip()

df[
    ["title", "description", "text"]
].sample(3)

,title,description,text
254387,crash [@ nsNSSSocketInfo::EnsureDocShellDependentStuffKnown()],crash nsNSSSocketInfo::EnsureDocShellDependentStuffKnown(); shortlived/gone but wanted to get this recorded in bugzilla.; mostly in 20090926 and 20090926 of SM; Fx; and TB.,crash [@ nsNSSSocketInfo::EnsureDocShellDependentStuffKnown()] crash nsNSSSocketInfo::EnsureDocShellDependentStuffKnown(); shortlived/gone but wanted to get this recorded in bugzilla.; mostly in 2...
183251,Mozilla crashed while waiting for Join selected cells operation to conclude in Editor [@ nsMenuFrame::OpenMenu],Mozilla crashed while I had a document open in the editor and while I was; waiting for a Join selected cells operation to conclude. I had selected three; table cells in a vertical column; summoned...,Mozilla crashed while waiting for Join selected cells operation to conclude in Editor [@ nsMenuFrame::OpenMenu] Mozilla crashed while I had a document open in the editor and while I was; waiting f...
123272,nsStr/nsString off-by-N bugs,While implementing javascript: URLs; I found a couple of bugs in xpcom/ds/nsStr*; and made the fixes yesterday; in the heat of stomach flu fever and in order to; get javascript: working for M9 (se...,nsStr/nsString off-by-N bugs While implementing javascript: URLs; I found a couple of bugs in xpcom/ds/nsStr*; and made the fixes yesterday; in the heat of stomach flu fever and in order to; get j...


In [ ]:
print(
    "Empty:",
    (df["text"].str.len() == 0).sum()
)
df = df[
    df["text"].str.len() > 0
].copy()

Empty: 4


In [ ]:
df["duplicate_of"].isna().value_counts()

,count
duplicate_of,
True,240374
False,80505


In [ ]:
num_duplicate = df["duplicate_of"].notna().sum()

print("Duplicate:", num_duplicate)
print("Total:", len(df))
print(
    "Rate:",
    num_duplicate / len(df)
)

Duplicate: 80505
Total: 320879
Rate: 0.25088896437597974


In [ ]:
duplicates = df[
    df["duplicate_of"].notna()
].copy()

duplicates[
    [
        "issue_id",
        "title",
        "duplicate_of",
        "created_at"
    ]
].head(10)

,issue_id,title,duplicate_of,created_at
1,14871,[Find] Find whole word only,269442.0,1999-09-24 21:49:34+00:00
23,134649,browser.urlbar.clickSelectsAll should default to false on Macintosh,409810.0,2002-04-01 16:39:09+00:00
30,153509,Allow search engines to test if they are the default,340604.0,2002-06-22 01:00:31+00:00
46,162344,attempting to open back button history menu in the second window generates another back button on the toolbar,165956.0,2002-08-12 18:32:44+00:00
57,167134,Customising toolbar breaks the throbber!,165823.0,2002-09-06 18:11:12+00:00
60,167183,Preferences > Privacy > Images == Wonky,161749.0,2002-09-06 21:38:44+00:00
64,167422,view source tries to download type application x-view-source from view-source:<url>,162593.0,2002-09-09 01:06:09+00:00
70,167969,Theme changes will take effect when you restart _Mozilla_.,167449.0,2002-09-11 14:45:36+00:00
83,169300,Pref UI to specify a proxy server for web navigation,161749.0,2002-09-17 23:57:05+00:00
88,169379,first Phoenix window ignored by nsIWindowMediator,169378.0,2002-09-18 07:57:39+00:00


In [ ]:
df["issue_id_str"] = (
    df["issue_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

df["duplicate_of_str"] = (
    df["duplicate_of"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

In [ ]:
df.loc[
    df["duplicate_of"].isna(),
    "duplicate_of_str"
] = None

In [ ]:
known_ids = set(df["issue_id_str"])
dup_df = df[
    df["duplicate_of_str"].notna()
].copy()
dup_df["target_exists"] = (
    dup_df["duplicate_of_str"]
    .isin(known_ids)
)
dup_df["target_exists"].value_counts()
print(
    dup_df["target_exists"].mean()
)

0.7469349729830446


In [ ]:
target_lookup = df[
    [
        "issue_id_str",
        "created_at",
        "text"
    ]
].rename(columns={
    "issue_id_str": "target_id",
    "created_at": "target_created_at",
    "text": "target_text"
})

In [ ]:
relations = dup_df.merge(
    target_lookup,
    left_on="duplicate_of_str",
    right_on="target_id",
    how="left"
)

In [ ]:
relations[
    [
        "issue_id_str",
        "created_at",
        "target_id",
        "target_created_at"
    ]
].head()

,issue_id_str,created_at,target_id,target_created_at
0,14871,1999-09-24 21:49:34+00:00,NaN,NaT
1,134649,2002-04-01 16:39:09+00:00,409810,2007-12-25 23:32:19+00:00
2,153509,2002-06-22 01:00:31+00:00,340604,2006-06-07 01:31:56+00:00
3,162344,2002-08-12 18:32:44+00:00,165956,2002-08-31 22:56:09+00:00
4,167134,2002-09-06 18:11:12+00:00,165823,2002-08-30 23:20:15+00:00


In [ ]:
relations["temporal_valid"] = (
    relations["target_created_at"]
    <
    relations["created_at"]
)
relations["temporal_valid"].value_counts()

,count
temporal_valid,
True,50477
False,30028


In [ ]:
strict_relations = relations[
    relations["temporal_valid"]
].copy()

print(
    "Strict temporal duplicate relations:",
    len(strict_relations)
)

Strict temporal duplicate relations: 50477


In [ ]:
df = df.sort_values(
    "created_at"
).reset_index(drop=True)

df[
    ["issue_id_str", "created_at"]
].head()

df[
    ["issue_id_str", "created_at"]
].tail()

,issue_id_str,created_at
320874,955877,2013-12-31 18:59:58+00:00
320875,955878,2013-12-31 20:02:55+00:00
320876,955890,2014-01-01 01:21:32+00:00
320877,955891,2014-01-01 03:03:01+00:00
320878,955893,2014-01-01 06:10:17+00:00


In [ ]:
df["year"] = df["created_at"].dt.year

df["year"].value_counts().sort_index()

year_stats = (
    df.groupby("year")
    .size()
    .reset_index(name="issues")
)

year_stats

,year,issues
0,1997,21
1,1998,1083
2,1999,12224
3,2000,19702
4,2001,21752
5,2002,23060
6,2003,14902
7,2004,19715
8,2005,21683
9,2006,19746


In [ ]:
q70 = df["created_at"].quantile(0.70)
q85 = df["created_at"].quantile(0.85)

print(q70)
print(q85)

train = df[
    df["created_at"] <= q70
].copy()

val = df[
    (df["created_at"] > q70)
    &
    (df["created_at"] <= q85)
].copy()

test = df[
    df["created_at"] > q85
].copy()

2010-04-07 20:08:16+00:00
2012-03-03 09:34:01.100000+00:00


In [ ]:
print("Train:", len(train))
print("Val:", len(val))
print("Test:", len(test))

Train: 224615
Val: 48132
Test: 48132


In [ ]:
print(
    train["created_at"].min(),
    train["created_at"].max()
)

print(
    val["created_at"].min(),
    val["created_at"].max()
)

print(
    test["created_at"].min(),
    test["created_at"].max()
)

1997-03-28 16:00:00+00:00 2010-04-07 19:55:25+00:00
2010-04-07 20:16:50+00:00 2012-03-03 09:03:17+00:00
2012-03-03 10:45:44+00:00 2014-01-01 06:10:17+00:00


In [ ]:
train_small = train.tail(10000).copy()

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

In [ ]:
train_matrix = vectorizer.fit_transform(
    train_small["text"]
)

In [ ]:
train_matrix.shape

(10000, 100000)

In [ ]:
query = val.iloc[0]

print(query["title"])

SVG/SMIL: keyTimes attribute should be honored when calcMode=discrete


In [ ]:
query_vector = vectorizer.transform(
    [query["text"]]
)

In [ ]:
scores = cosine_similarity(
    query_vector,
    train_matrix
).ravel()

In [ ]:
top_idx = scores.argsort()[::-1][:10]

In [ ]:
results = train_small.iloc[top_idx].copy()

results["score"] = scores[top_idx]

results[
    [
        "issue_id_str",
        "title",
        "created_at",
        "score"
    ]
]

,issue_id_str,title,created_at,score
220767,544855,SMIL: Discrete to-animation should only visit the to value,2010-02-08 12:03:58+00:00,0.250570
223716,555026,SVG/SMIL: keyTimes attribute should be ignored when calcMode=paced,2010-03-25 20:02:21+00:00,0.226962
224274,556841,Assertions: Wrong number of elements in from value; invalid array index; etc. with animateTransform by-animation w/ calcMode=paced,2010-04-02 19:19:20+00:00,0.128552
219784,541882,Add support for SMIL animation of the preserveAspectRatio attribute in SVG,2010-01-24 23:39:35+00:00,0.124098
214836,525721,<input type=file multiple> value attribute only shows first file,2009-11-01 11:06:01+00:00,0.122939
220969,545550,SVG/SMIL: Setters for SVG attr base value shouldnt directly modify animated value; when animations are active,2010-02-11 01:01:31+00:00,0.117777
217764,535915,Intermittent failure in anim-discrete-values-1.svg,2009-12-19 01:43:12+00:00,0.116268
224167,556404,SMIL Interpolation doesnt work on mapped attributes when units are left off,2010-03-31 22:50:06+00:00,0.108240
219786,541884,Add support for SMIL animation of the viewBox attribute in SVG.,2010-01-25 00:08:42+00:00,0.103895
222667,551670,type attribute for button elements should be submit when type is invalid or missing,2010-03-11 13:26:49+00:00,0.095286


In [ ]:
def reciprocal_rank(
    retrieved_ids,
    relevant_ids
):
    relevant_ids = set(relevant_ids)

    for rank, issue_id in enumerate(
        retrieved_ids,
        start=1
    ):
        if issue_id in relevant_ids:
            return 1.0 / rank

    return 0.0

In [ ]:
def recall_at_k(
    retrieved_ids,
    relevant_ids,
    k
):
    relevant_ids = set(relevant_ids)

    top_k = set(
        retrieved_ids[:k]
    )

    return int(
        len(
            top_k.intersection(
                relevant_ids
            )
        ) > 0
    )

In [ ]:
ground_truth = {
    row["issue_id_str"]:
        row["duplicate_of_str"]
    for _, row in strict_relations.iterrows()
}

In [ ]:
list(ground_truth.items())[:5]

[('167134', '165823'),
 ('167183', '161749'),
 ('167969', '167449'),
 ('169300', '161749'),
 ('169379', '169378')]

In [ ]:
val_duplicates = val[
    val["issue_id_str"].isin(
        ground_truth.keys()
    )
].copy()

In [ ]:
val_duplicates = (
    val_duplicates
    .sort_values("created_at")
)

print(
    "Validation duplicate queries:",
    len(val_duplicates)
)

Validation duplicate queries: 5851


In [ ]:
history = train.copy()

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

history_matrix = vectorizer.fit_transform(
    history["text"]
)

In [ ]:
metrics = []

for _, query in val_duplicates.head(100).iterrows():

    q_vec = vectorizer.transform(
        [query["text"]]
    )

    scores = cosine_similarity(
        q_vec,
        history_matrix
    ).ravel()

    top_idx = scores.argsort()[::-1][:10]

    retrieved = (
        history
        .iloc[top_idx]["issue_id_str"]
        .tolist()
    )

    target = ground_truth[
        query["issue_id_str"]
    ]

    relevant = [target]

    metrics.append({
        "query_id": query["issue_id_str"],
        "recall@1": recall_at_k(
            retrieved,
            relevant,
            1
        ),
        "recall@5": recall_at_k(
            retrieved,
            relevant,
            5
        ),
        "recall@10": recall_at_k(
            retrieved,
            relevant,
            10
        ),
        "rr": reciprocal_rank(
            retrieved,
            relevant
        )
    })

In [ ]:
metrics_df = pd.DataFrame(metrics)

In [ ]:
metrics_df[
    [
        "recall@1",
        "recall@5",
        "recall@10",
        "rr"
    ]
].mean()

,0
recall@1,0.0400
recall@5,0.1200
recall@10,0.1300
rr,0.0665


In [ ]:
query = val_duplicates.iloc[0]

print("QUERY")
print(query["issue_id_str"])
print(query["title"])

print("\nGROUND TRUTH")
print(
    ground_truth[
        query["issue_id_str"]
    ]
)

QUERY
557919
[OS/2] unresolved externals related to WebGL after landing of bug533637

GROUND TRUTH
557660


In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

vectorizer.fit(train["text"])

TfidfVectorizer(max_features=100000, min_df=2, ngram_range=(1, 2),
                sublinear_tf=True)

In [ ]:
all_matrix = vectorizer.transform(
    df["text"]
)

In [ ]:
id_to_pos = {
    issue_id: i
    for i, issue_id
    in enumerate(df["issue_id_str"])
}

In [ ]:
test_queries = test[
    test["issue_id_str"].isin(
        ground_truth.keys()
    )
].sort_values("created_at")

In [ ]:
results = []

for _, query in test_queries.iterrows():

    query_pos = id_to_pos[
        query["issue_id_str"]
    ]

    history_mask = (
        df["created_at"]
        <
        query["created_at"]
    )

    history_positions = np.where(
        history_mask.values
    )[0]

    if len(history_positions) == 0:
        continue

    q_vec = all_matrix[
        query_pos
    ]

    history_matrix = all_matrix[
        history_positions
    ]

    scores = cosine_similarity(
        q_vec,
        history_matrix
    ).ravel()

    top_n = min(
        10,
        len(scores)
    )

    local_top_idx = (
        scores
        .argsort()[::-1][:top_n]
    )

    global_top_idx = (
        history_positions[
            local_top_idx
        ]
    )

    retrieved_ids = (
        df.iloc[
            global_top_idx
        ]["issue_id_str"]
        .tolist()
    )

    target = ground_truth[
        query["issue_id_str"]
    ]

    relevant = [target]

    results.append({
        "query_id":
            query["issue_id_str"],

        "target":
            target,

        "recall@1":
            recall_at_k(
                retrieved_ids,
                relevant,
                1
            ),

        "recall@5":
            recall_at_k(
                retrieved_ids,
                relevant,
                5
            ),

        "recall@10":
            recall_at_k(
                retrieved_ids,
                relevant,
                10
            ),

        "rr":
            reciprocal_rank(
                retrieved_ids,
                relevant
            )
    })

In [ ]:
result_df = pd.DataFrame(results)

In [ ]:
summary = {
    "queries": len(result_df),

    "Recall@1":
        result_df["recall@1"].mean(),

    "Recall@5":
        result_df["recall@5"].mean(),

    "Recall@10":
        result_df["recall@10"].mean(),

    "MRR":
        result_df["rr"].mean()
}

summary

{'queries': 4081,
 'Recall@1': np.float64(0.10757167360940946),
 'Recall@5': np.float64(0.18426856162705219),
 'Recall@10': np.float64(0.22739524626317079),
 'MRR': np.float64(0.14134977032551155)}

In [ ]:
assert (
    train["created_at"].max()
    <
    val["created_at"].min()
)

assert (
    val["created_at"].max()
    <
    test["created_at"].min()
)

print("Temporal split OK")

Temporal split OK


In [ ]:
assert (
    strict_relations["target_created_at"]
    <
    strict_relations["created_at"]
).all()

print("Duplicate temporal relations OK")

Duplicate temporal relations OK


In [ ]:
MODEL_INPUT_COLUMNS = [
    "title_clean",
    "description_clean"
]

In [ ]:
df.to_parquet(
    "/content/issues_processed.parquet",
    index=False
)

In [ ]:
strict_relations.to_parquet(
    "/content/duplicate_relations.parquet",
    index=False
)

In [ ]:
train.to_parquet(
    "/content/train.parquet",
    index=False
)

val.to_parquet(
    "/content/validation.parquet",
    index=False
)

test.to_parquet(
    "/content/test.parquet",
    index=False
)

In [ ]:
result_df.to_csv(
    "/content/tfidf_test_results.csv",
    index=False
)

In [ ]:
summary_df = pd.DataFrame(
    [summary]
)

summary_df.to_csv(
    "/content/tfidf_summary.csv",
    index=False
)

In [ ]:
print("=" * 60)
print("PHASE 1 — DATA & BENCHMARK")
print("=" * 60)

print("\nDATASET")
print("Total issues:", len(df))
print(
    "Firefox:",
    (df["project"] == "firefox").sum()
)
print(
    "Mozilla Core:",
    (df["project"] == "mozillacore").sum()
)

print("\nDUPLICATES")
print(
    "Raw duplicate relations:",
    len(dup_df)
)
print(
    "Target exists:",
    dup_df["target_exists"].sum()
)
print(
    "Strict temporal relations:",
    len(strict_relations)
)

print("\nSPLIT")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

print("\nTF-IDF BASELINE")
print(
    f"Recall@1:  {summary['Recall@1']:.4f}"
)
print(
    f"Recall@5:  {summary['Recall@5']:.4f}"
)
print(
    f"Recall@10: {summary['Recall@10']:.4f}"
)
print(
    f"MRR:       {summary['MRR']:.4f}"
)

PHASE 1 — DATA & BENCHMARK

DATASET
Total issues: 320879
Firefox: 115812
Mozilla Core: 205067

DUPLICATES
Raw duplicate relations: 80505
Target exists: 60132
Strict temporal relations: 50477

SPLIT
Train: 224615
Validation: 48132
Test: 48132

TF-IDF BASELINE
Recall@1:  0.1076
Recall@5:  0.1843
Recall@10: 0.2274
MRR:       0.1413


In [ ]:
missing_targets = dup_df[
    ~dup_df["target_exists"]
].copy()

print("Missing targets:", len(missing_targets))

missing_targets[
    [
        "project",
        "issue_id_str",
        "duplicate_of_str",
        "title"
    ]
].head(20)

Missing targets: 20373


,project,issue_id_str,duplicate_of_str,title
1,firefox,14871,269442,[Find] Find whole word only
64,firefox,167422,162593,view source tries to download type application x-view-source from view-source:<url>
101,firefox,170029,25894,Respect Windows Access Keys display pref (default: only show the access keys when you hit alt)
110,firefox,170362,102831,Ctrl-W does not close the current tab
128,firefox,170571,20618,Mousewheel does not scroll web page
133,firefox,170630,170545,Scroll bar becomes blank
149,firefox,170844,170545,scroll bar button arrows magically move
157,firefox,170961,164878,no resume choice in download manager
218,firefox,171467,8648,Run check on bookmark validity
226,firefox,171502,105547,scripts opens windows instead of tabs


In [ ]:
missing_targets[
    "project"
].value_counts()

,count
project,
firefox,10669
mozillacore,9704


In [ ]:
issue_projects = (
    df.groupby("issue_id_str")["project"]
      .apply(list)
      .to_dict()
)

missing_targets["found_projects"] = (
    missing_targets["duplicate_of_str"]
    .map(issue_projects)
)

missing_targets[
    missing_targets["found_projects"].notna()
][
    [
        "project",
        "issue_id_str",
        "duplicate_of_str",
        "found_projects"
    ]
].head(20)

,project,issue_id_str,duplicate_of_str,found_projects


In [ ]:
invalid_temporal = relations[
    relations["target_created_at"].notna()
    &
    ~relations["temporal_valid"]
].copy()

print(
    "Temporal invalid:",
    len(invalid_temporal)
)

Temporal invalid: 9655


In [ ]:
invalid_temporal[
    [
        "issue_id_str",
        "created_at",
        "target_id",
        "target_created_at"
    ]
].head(20)

,issue_id_str,created_at,target_id,target_created_at
1,134649,2002-04-01 16:39:09+00:00,409810,2007-12-25 23:32:19+00:00
2,153509,2002-06-22 01:00:31+00:00,340604,2006-06-07 01:31:56+00:00
3,162344,2002-08-12 18:32:44+00:00,165956,2002-08-31 22:56:09+00:00
12,170216,2002-09-22 21:24:45+00:00,280190,2005-01-28 13:06:36+00:00
14,170459,2002-09-24 04:50:16+00:00,170463,2002-09-24 05:45:01+00:00
18,170526,2002-09-24 16:11:50+00:00,170623,2002-09-25 01:29:53+00:00
36,171184,2002-09-27 15:32:56+00:00,171196,2002-09-27 15:48:32+00:00
37,171185,2002-09-27 15:33:01+00:00,171196,2002-09-27 15:48:32+00:00
38,171187,2002-09-27 15:33:05+00:00,171196,2002-09-27 15:48:32+00:00
39,171189,2002-09-27 15:33:10+00:00,171196,2002-09-27 15:48:32+00:00
